# Project 1: System inventory

## 1. What can happen?

An AI helper has no named owner, while another is due for review.

**Example** The leave request helper's owner field is blank. Who should review it?

## 2. Our idea

An inventory is a list of applications with their purposes, owners, and next
review dates. We load three fictional records and run a small Python check.
This notebook makes no model calls and requires no API key.

### How we do it

1. Read the three records and choose a fixed sample date.
2. Check required fields, then see whether each review is due.
3. Fill one fictional owner field in a copy and compare the same records.

## 3. How it helps

We can see what needs attention. A complete record only passes this documentation
check. It neither proves the information is true nor authorizes deployment.

## 4. Input and setup

Run uv sync --locked from the learning-governance repository root first.
Open 1_system_inventory as the VS Code folder and select the root
.venv/Scripts/python.exe kernel. Run one code cell at a time.

The first cell checks the lesson folder and shows which Python is running.

In [ ]:
import json
import sys
from datetime import date
from pathlib import Path

project_dir = Path.cwd()
data_path = project_dir / "docs" / "dataset" / "systems.json"
assert data_path.is_file(), "Open 1_system_inventory as the VS Code folder."
print("Lesson folder:", project_dir.name)
print("Python:", sys.executable)

### Step 2: Read the records

The file docs/dataset/systems.json is fictional teaching data.
HR-001 has all fields and a future review, HR-002 has a blank owner, and
HR-003 is due on the sample date we will use.

Run this cell and look at the owner and next_review_on fields.
Nothing is sent to a model.

In [ ]:
systems = json.loads(data_path.read_text(encoding="utf-8"))
print(json.dumps(systems, indent=2))

### Step 3: Understand the helper and choose the sample date

Read [tools/inventory.py](tools/inventory.py) before importing it.
The check_system function first finds missing, non-text, or blank required
fields. If any are missing, its status is incomplete. Otherwise it compares
the next review date with the sample date. A review today or earlier is due.

The check_inventory function applies that check to every record without
changing the data. The show_results function prints each status and reason.

The source examples use unique IDs and valid YYYY-MM-DD dates. The fixed
sample date, 2026-11-15, makes the comparison reproducible on later runs.
This is a teaching setting, rather than an automatic check of today's date.

In [ ]:
from tools.inventory import check_inventory, show_results

as_of = date(2026, 11, 15)
print("Fixed sample date:", as_of)

### Step 4: Run the baseline

Run the original records without editing them.
Expect HR-001 to be complete, HR-002 incomplete, and HR-003 review_due.

The due date is inclusive: a review scheduled for 2026-11-15 is due when
our sample date is also 2026-11-15. Each result includes a reason.

In [ ]:
baseline = check_inventory(systems, as_of)
show_results(baseline)

### Step 5: Fill one fictional owner field

Use deepcopy to make a separate in-memory copy of the records.
Change only the second record, HR-002, by assigning the fictional role
Leave operations. Keep the same dates and checker.

Expect HR-002 to become complete. HR-003 should still be review_due.
This is an example field edit, not a real assignment or an owner's acceptance.
The source JSON and baseline results stay unchanged.

In [ ]:
from copy import deepcopy

candidate_systems = deepcopy(systems)
candidate_systems[1]["owner"] = "Leave operations"
candidate = check_inventory(candidate_systems, as_of)
show_results(candidate)

### Step 6: Compare the same records

Read the before and after status for each application.
Only HR-002 should change. The final line shows that the original owner is
still an empty string. This notebook writes no files.

In [ ]:
for before, after in zip(baseline, candidate):
    print(f"{before['system_id']}: {before['status']} -> {after['status']}")

print("Original HR-002 owner:", repr(systems[1]["owner"]))

### Explain the result

The baseline has one complete record, one incomplete record, and one review due.
After the owner edit, two records are complete and one still needs review.

Explain why the due review remains and why a present owner field cannot prove
that the owner accepted responsibility. Also consider why checking these fields
does not approve the assistant for deployment.

No results or human judgments have been saved. Pause here before choosing
Project 2. Restart the kernel before rerunning after a helper edit.